# VELOVI Notebook

## By Josh Navarro

### Notes:
VeloVi is a variational autoencoder...

In [ ]:
# Imports
import pandas as pd
import numpy as np
from velovi import VELOVI, preprocess_data
import anndata as ad
import scanpy as sc
import scvelo as scv
import torch

In [4]:
# Data
adata = ad.read_h5ad("C:/Users/joshu/Desktop/RNAVelo_Tools_Comparison/data/janssens_2025_CellRanger_Velocyto_counts_eLNP.h5ad")
# Check the data
adata

AnnData object with n_obs × n_vars = 21327 × 31353
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'subsets_RBC_sum', 'subsets_RBC_detected', 'subsets_RBC_percent', 'subsets_COVID_sum', 'subsets_COVID_detected', 'subsets_COVID_percent', 'total', 'sizeFactor', 'nCount_RNA.y', 'nFeature_RNA.y', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.0', 'SCT_snn_res.0.1', 'SCT_snn_res.0.2', 'SCT_snn_res.0.3', 'SCT_snn_res.0.4', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6', 'SCT_snn_res.0.7', 'SCT_snn_res.0.9', 'SCT_snn_res.1', 'SCT_clusters', 'fzero', 'pca.drop', 'pANN_0.25_0.13_181', 'DFPrediction', 'nCount_ADT', 'nFeature_ADT', 'ADT_snn_res.0.8', 'ADT_snn_res.0', 'ADT_snn_res.0.1', 'ADT_snn_res.0.2', 'ADT_snn_res.0.3', 'ADT_snn_res.0.4', 'ADT_snn_res.0.5', 'ADT_snn_res.0.6', 'ADT_snn_res.0.7', 'ADT_snn_res.0.9', 'ADT_snn_res.1', 'ADT_clusters', 'nCount_HTO', 'nFeature_HTO', 'HT

In [5]:
# Save raw counts first
adata.layers["counts"] = adata.X.copy()

# Then filter and normalize
scv.pp.filter_and_normalize(adata, min_shared_counts=30)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor='seurat_v3',
    layer="counts",
    subset=True
)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

Filtered out 24378 genes that are detected 30 counts (shared).
Normalized count data: X, spliced, unspliced.


C:\Users\joshu\AppData\Local\Temp\ipykernel_20472\3313257732.py:14: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
c:\Users\joshu\miniconda3\envs\velovi_env\Lib\site-packages\scvelo\preprocessing\moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(
c:\Users\joshu\miniconda3\envs\velovi_env\Lib\site-packages\scvelo\preprocessing\neighbors.py:233: DeprecationWarning: Automatic computation of PCA is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute PCA with Scanpy first.
  _set_pca(adata=adata, n_pcs=n_pcs, use_highly_variable=use_highly_variable)


computing neighbors
    finished (0:01:16) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:05) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)


In [8]:
adata = preprocess_data(adata)

computing velocities
    finished (0:00:02) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)


In [ ]:
VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = VELOVI(adata)
vae.train(accelerator="cpu",
          max_epochs=200,
          early_stopping=True,
          batch_size=512)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Training:   0%|          | 0/500 [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Exception raised during training. <class 'NameError'> 1


SystemExit: 1

In [ ]:
def add_velovi_outputs_to_adata(adata, vae):
    latent_time = vae.get_latent_time(n_samples=25)
    velocities = vae.get_velocity(n_samples=25, velo_statistic="mean")

    t = latent_time
    scaling = 20 / t.max(0)

    adata.layers["velocity"] = velocities / scaling
    adata.layers["latent_time_velovi"] = latent_time

    adata.var["fit_alpha"] = vae.get_rates()["alpha"] / scaling
    adata.var["fit_beta"] = vae.get_rates()["beta"] / scaling
    adata.var["fit_gamma"] = vae.get_rates()["gamma"] / scaling
    adata.var["fit_t_"] = (
        torch.nn.functional.softplus(vae.module.switch_time_unconstr).detach().cpu().numpy()
    ) * scaling
    adata.layers["fit_t"] = latent_time.values * scaling.values[np.newaxis, :]  # fix here
    adata.var["fit_scaling"] = 1.0

add_velovi_outputs_to_adata(adata, vae)

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
sc.tl.umap(adata)

In [ ]:
adata.obs['Experimental_condition'] = adata.obs['Condition'].str.split(r" [0-9]").str[0]

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis="umap", color="newClustersTmp")

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis="umap", color="Experimental_condition")